# Assignment 1 — training pipeline

Bank Marketing (UCI): predict whether a client subscribes to a term deposit, so the
sales team can decide **whom to call next** under a limited call-center capacity.
See `REPORT.md` for the business framing.

> Runs end-to-end via `uv run python main.py train` (converted to a plain script),
> so: use `print(...)` instead of bare expressions / `display(...)`, and save plots
> to files instead of `plt.show()`.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("data/bank-additional-full.csv")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 2. Load the data

The file is `;`-separated. The target `y` is `"yes"`/`"no"` → we encode it as 1/0.

In [2]:
raw = pd.read_csv(DATA_PATH, sep=";")
raw["y"] = (raw["y"] == "yes").astype(int)

print(raw.shape)
print(raw.head())
print(f"Positive rate: {raw['y'].mean():.3f}")

(41188, 21)
   age        job  marital    education  default housing loan    contact month day_of_week  duration  campaign  pdays  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   may         mon       261         1    999   
1   57   services  married  high.school  unknown      no   no  telephone   may         mon       149         1    999   
2   37   services  married  high.school       no     yes   no  telephone   may         mon       226         1    999   
3   40     admin.  married     basic.6y       no      no   no  telephone   may         mon       151         1    999   
4   56   services  married  high.school       no      no  yes  telephone   may         mon       307         1    999   

   previous     poutcome  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed  y  
0         0  nonexistent           1.1          93.994          -36.4      4.857       5191.0  0  
1         0  nonexistent           1.1          93.994         

## 3. Reconstruct the calendar date

The rows are ordered by date (May 2008 – Nov 2010, per the UCI documentation), but
there is no `year` column. Every time the month number goes *down* compared with the
previous row, a new year has started. This gives us a `year` and a `period`
(year-month) that we need for the temporal split.

In [3]:
MONTHS = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
month_num = raw["month"].map({m: i for i, m in enumerate(MONTHS, start=1)})

df = raw.copy()
df["year"] = 2008 + (month_num.diff() < 0).cumsum()
df["period"] = pd.PeriodIndex.from_fields(year=df["year"], month=month_num, freq="M")

print(df.groupby("year")["y"].agg(rows="size", positive_rate="mean").round(3))

       rows  positive_rate
year                      
2008  27690          0.048
2009  11440          0.195
2010   2058          0.521


## 4. Data-quality overview

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("'unknown' values per column:")
print((df == "unknown").sum().loc[lambda s: s > 0])
print()
print(f"pdays == 999 (never contacted before): {(df['pdays'] == 999).mean():.1%}")

Duplicate rows: 12

'unknown' values per column:
job           330
marital        80
education    1731
default      8597
housing       990
loan          990
dtype: int64

pdays == 999 (never contacted before): 96.3%


In [5]:
by_period = df.groupby("period").agg(
    calls=("y", "size"),
    positive_rate=("y", "mean"),
    euribor3m=("euribor3m", "mean"),
)
by_period["cum_share_of_rows"] = by_period["calls"].cumsum() / len(df)
print(by_period.round(3).to_string())

         calls  positive_rate  euribor3m  cum_share_of_rows
period                                                     
2008-05   7763          0.031      4.858              0.188
2008-06   4374          0.043      4.932              0.295
2008-07   6685          0.061      4.962              0.457
2008-08   5175          0.052      4.965              0.583
2008-10     67          0.627      4.924              0.584
2008-11   3616          0.053      4.115              0.672
2008-12     10          0.100      3.493              0.672
2009-03    282          0.447      1.646              0.679
2009-04   2458          0.180      1.412              0.739
2009-05   5794          0.090      1.294              0.879
2009-06    715          0.369      1.258              0.897
2009-07    178          0.371      1.025              0.901
2009-08    770          0.343      0.866              0.920
2009-09    267          0.397      0.783              0.926
2009-10    447          0.403      0.735

## 5. Leakage: drop `duration`

`duration` is only known *after* the call (see REPORT.md → Business framing), so it
cannot be an input to a model that decides whom to call.

In [6]:
df = df.drop(columns=["duration"])

## 6. Data-quality decisions

- **Duplicates**: drop the 12 exact duplicate rows of the original file (negligible
  share, avoids counting the same call twice). They must be detected *before*
  dropping `duration`, otherwise ~1,800 distinct calls look identical.
- **`"unknown"`**: kept as its own category. Not knowing is informative (e.g.
  `default = unknown` subscribes 5.2% vs. 12.9% for `default = no`), and new clients
  with `"unknown"` will keep arriving in production.
- **`pdays`**: 999 is a sentinel for "never contacted", not a number of days. A linear
  model would read it as "a very long time". My first choice was to replace it with a
  binary flag `previously_contacted`. However, the cross-tab below shows the flag is
  almost a copy of `poutcome`: all 1,373 `poutcome = success` clients are flagged,
  plus only 142 `failure` clients. `poutcome` already says "never contacted / failure / success",
  so it contains more information than the flag. To avoid redundant columns (which
  split one effect across two weights), I drop both `pdays` and the flag and keep
  `poutcome`.

In [7]:
# Detect duplicates on the ORIGINAL columns: after dropping `duration`, distinct calls
# with the same client profile would wrongly look identical (~1,800 rows instead of 12).
n_before = len(df)
df = df.loc[~raw.duplicated()].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} duplicate rows -> {len(df)} rows")

previously_contacted = (df["pdays"] != 999).astype(int).rename("previously_contacted")
print(pd.crosstab(df["poutcome"], previously_contacted))
print()
print(df.groupby("poutcome")["y"].agg(rows="size", positive_rate="mean").round(3))
df = df.drop(columns=["pdays"])

Dropped 12 duplicate rows -> 41176 rows
previously_contacted      0     1
poutcome                         
failure                4110   142
nonexistent           35551     0
success                   0  1373

              rows  positive_rate
poutcome                         
failure       4252          0.142
nonexistent  35551          0.088
success       1373          0.651


## 7. Temporal split

The model will be used to pick whom to call in **future** campaigns, so we train on
the past and evaluate on later periods:

| Set | Periods | Why |
|---|---|---|
| Train | May 2008 – May 2009 | All the history available before the regime change |
| Validation | Jun – Dec 2009 | Same "new regime" as the test set (low Euribor, high conversion), so the threshold chosen here transfers to test |
| Test | 2010 | The "next year": touched only once, at the end |

In [8]:
TRAIN_END = pd.Period("2009-05", freq="M")
VAL_END = pd.Period("2009-12", freq="M")

split = np.where(df["period"] <= TRAIN_END, "train",
                 np.where(df["period"] <= VAL_END, "val", "test"))
df["split"] = split

summary = df.groupby("split", sort=False).agg(
    rows=("y", "size"),
    share=("y", lambda s: len(s) / len(df)),
    positive_rate=("y", "mean"),
    first=("period", "min"),
    last=("period", "max"),
)
print(summary.round(3))

        rows  share  positive_rate    first     last
split                                               
train  36214  0.879          0.067  2008-05  2009-05
val     2904  0.071          0.391  2009-06  2009-12
test    2058  0.050          0.521  2010-03  2010-11


C:\Users\Eduardo Alfaro\AppData\Local\Temp\ipykernel_55472\2794030103.py:15: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(summary.round(3))


## 8. Feature engineering

Logistic regression is linear: each feature gets one weight, so the effect of a
feature can only go up or down monotonically. The transformations below shape the
inputs so the model can use the patterns found in the **training** data.

| Feature | Decision | Why |
|---|---|---|
| `month` | **Dropped** | Its effect is the year/crisis regime, not seasonality (May: 3% in 2008, 9% in 2009, 58% in 2010), and September never appears in train. |
| `age` | **Age bands** (one-hot) | U-shaped relation (<=25: 10.7%, 41-50: 5.4%, >60: 36.2%) that a single linear weight cannot represent. |
| `campaign` | **`log(1 + campaign)`** | Long tail (median 2, max 56). The log gives diminishing returns to extra calls and stops a few extreme clients from dominating the weight. |
| Macro indicators | **Keep only `euribor3m`** | `euribor3m`, `emp.var.rate` and `nr.employed` are 0.96-0.98 correlated (multicollinearity -> unstable weights). One indicator keeps the economic context. |
| `housing` / `loan` "unknown" | **One flag `loans_unknown`** | The same 990 clients are "unknown" in both columns, so the two one-hot columns were identical. The flag holds that information once; `housing` and `loan` become yes/no. |

Encoding: one-hot for categoricals (first level dropped as the reference category, so
the weights are identifiable and comparable across the three implementations), then
**every column is standardised, one-hot columns included**. Without this, rare
categories (e.g. `age_band = >60`, 163 rows) are columns that are almost always 0, so
their gradient is tiny and plain gradient descent needs a huge number of steps along
that direction. With unscaled one-hot columns the manual PyTorch loop and
scikit-learn gave predictions up to 0.54 apart after 2,000 epochs; after scaling,
they differ by ~0.002. **The encoder and scaler are fitted on the training set only.**

In [9]:
AGE_BINS = [-np.inf, 25, 30, 40, 50, 60, np.inf]
AGE_LABELS = ["<=25", "26-30", "31-40", "41-50", "51-60", ">60"]


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Row-wise transformations (no statistics learned from data, so no leakage risk)."""
    out = data.copy()
    out["age_band"] = pd.cut(out["age"], bins=AGE_BINS, labels=AGE_LABELS).astype(str)
    out["log_campaign"] = np.log1p(out["campaign"])
    # housing/loan are "unknown" for exactly the same clients -> one flag instead of two identical columns.
    out["loans_unknown"] = (out["housing"] == "unknown").astype(int)
    out["housing"] = out["housing"].replace("unknown", "no")
    out["loan"] = out["loan"].replace("unknown", "no")
    return out


assert ((df["housing"] == "unknown") == (df["loan"] == "unknown")).all()
df = engineer_features(df)

print(df[df["split"] == "train"].groupby("age_band", sort=False)["y"]
      .agg(rows="size", positive_rate="mean").reindex(AGE_LABELS).round(3))

           rows  positive_rate
age_band                      
<=25       1185          0.107
26-30      4879          0.084
31-40     14915          0.065
41-50      9518          0.054
51-60      5554          0.063
>60         163          0.362


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAL = ["job", "marital", "education", "default", "housing", "loan",
               "contact", "poutcome", "day_of_week", "age_band"]
NUMERIC = ["log_campaign", "previous", "loans_unknown", "euribor3m"]
TARGET = "y"

preprocessor = ColumnTransformer([
    ("cat", make_pipeline(
        OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
        StandardScaler(),  # scale one-hot columns too: needed for gradient descent to converge
    ), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
])

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

# Fit on train only, then apply the same transformation to val and test.
X_train = preprocessor.fit_transform(train_df[CATEGORICAL + NUMERIC]).astype(np.float32)
X_val = preprocessor.transform(val_df[CATEGORICAL + NUMERIC]).astype(np.float32)
X_test = preprocessor.transform(test_df[CATEGORICAL + NUMERIC]).astype(np.float32)
y_train, y_val, y_test = (d[TARGET].to_numpy(np.float32) for d in (train_df, val_df, test_df))

FEATURE_NAMES = list(preprocessor.get_feature_names_out())
print(f"{len(FEATURE_NAMES)} model features")
print("X_train", X_train.shape, "| X_val", X_val.shape, "| X_test", X_test.shape)

# Categories present in val/test but never seen in train would be silently encoded as all zeros.
for col in CATEGORICAL:
    unseen = set(pd.concat([val_df, test_df])[col]) - set(train_df[col])
    if unseen:
        print(f"Unseen categories in {col}: {unseen}")

41 model features
X_train (36214, 41) | X_val (2904, 41) | X_test (2058, 41)


In [11]:
import joblib

joblib.dump(preprocessor, ARTIFACTS / "preprocessor.joblib")
# Cleaned dataset with the split label: used by the dashboard for distributions.
df.drop(columns=["period"]).to_csv(ARTIFACTS / "dataset_with_split.csv", index=False)
print("Saved preprocessor and dataset to", ARTIFACTS)

Saved preprocessor and dataset to artifacts


## 9. Evaluation metrics (from the business framing)

- **Main metric: precision at 25% capacity per campaign.** In each month (campaign),
  the call center calls the 25% of clients with the highest score. Precision = share of
  those calls that end in a subscription. Ranking *within* a month also means every
  client being compared has the same `euribor3m`, so only the client profile decides
  the order.
- **Lift** = precision / positive rate of the set. The positive rate differs a lot
  between periods (val 39%, test 52%), so lift says how much better than calling at
  random the model is, independently of the period.
- **Recall at capacity**: share of all subscribers that we actually reach.
- **Business cost** = FP x 5 EUR + FN x 100 EUR for the calls made at capacity.
- **AUC** and **log loss** to compare the three implementations on the full probability
  output, not only on the top 25%.

In [12]:
from sklearn.metrics import log_loss, roc_auc_score

CAPACITY = 0.25
COST_FP = 5     # EUR: calling a client who does not subscribe
COST_FN = 100   # EUR: lost margin from a subscriber we did not call


def select_top_per_period(scores, periods, capacity=CAPACITY):
    """Boolean mask: the top `capacity` share of clients by score within each period."""
    d = pd.DataFrame({"score": np.asarray(scores), "period": np.asarray(periods)})
    rank = d.groupby("period")["score"].rank(method="first", ascending=False)
    size = d.groupby("period")["score"].transform("size")
    return (rank <= np.round(capacity * size)).to_numpy()


def evaluate(y, proba, periods, scores=None):
    """Business + statistical metrics. `scores` (for ranking) defaults to the probabilities."""
    y = np.asarray(y)
    called = select_top_per_period(proba if scores is None else scores, periods)
    tp = int((called & (y == 1)).sum())
    fp = int((called & (y == 0)).sum())
    fn = int((~called & (y == 1)).sum())
    precision = tp / called.sum()
    return {
        "precision@25%": precision,
        "lift": precision / y.mean(),
        "recall@25%": tp / (y == 1).sum(),
        "business_cost_eur": fp * COST_FP + fn * COST_FN,
        "auc": roc_auc_score(y, proba),
        "log_loss": log_loss(y, np.clip(proba, 1e-7, 1 - 1e-7)),
    }

## 10. Naive baseline

The sales team without a model: call 25% of each month's clients **at random**, and
predict the training positive rate as everyone's probability. Its expected precision
is just the positive rate of the period (lift = 1). Accuracy is not used: predicting
"no" for everyone already gets 93% accuracy on train.

In [13]:
rng = np.random.default_rng(SEED)
base_rate = y_train.mean()

baseline = {}
for name, y, frame in [("val", y_val, val_df), ("test", y_test, test_df)]:
    constant_proba = np.full(len(y), base_rate)
    baseline[name] = evaluate(y, constant_proba, frame["period"], scores=rng.random(len(y)))
print(pd.DataFrame(baseline).T.round(3))

      precision@25%   lift  recall@25%  business_cost_eur  auc  log_loss
val           0.369  0.944       0.236            88990.0  0.5     1.098
test          0.509  0.976       0.244            82365.0  0.5     1.442


## 11. Three implementations of the same logistic regression

All three minimise **the same objective** on the same standardised training data:

$$\mathcal{L}(w, b) = \frac{1}{n}\sum_i \big[\log(1 + e^{z_i}) - y_i z_i\big] + \frac{\lambda}{2}\lVert w \rVert^2,
\qquad z_i = x_i^\top w + b,\qquad \lambda = \frac{1}{C\,n}$$

scikit-learn's `C` multiplies the summed loss, so dividing its objective by `C·n` gives
exactly this mean loss with `λ = 1/(C·n)`. The intercept `b` is not penalised, as in
scikit-learn. Because the objective is strictly convex (thanks to the L2 term), it has
a single minimum, and all three methods should land on the same weights.

Hyperparameters (chosen on the validation set, see REPORT.md):
- `C = 1` (scikit-learn's default). Validation precision@25% was flat for any
  `C >= 0.001` (53.0%-53.7%, within noise), so I kept the standard value.
- Learning rate `0.5` for both PyTorch versions. At 0.01-0.1 gradient descent had not
  converged after 2,000 epochs; from 0.5 up the predictions match scikit-learn.
- 10,000 epochs. With `C = 1` the penalty is weak, and `previous` and
  `poutcome_nonexistent` are strongly correlated (-0.975), which leaves an almost flat
  direction in the loss. Gradient descent moves along it slowly: at 3,000 epochs the
  test probabilities still differed from scikit-learn by up to 0.018, at 10,000 by
  less than 0.001 (about 9 s of training).

In [14]:
import torch
from sklearn.linear_model import LogisticRegression

C = 1.0
LR = 0.5
EPOCHS = 10_000
N_TRAIN, N_FEATURES = X_train.shape
LAMBDA = 1.0 / (C * N_TRAIN)

torch.manual_seed(SEED)
Xt_train, yt_train = torch.from_numpy(X_train), torch.from_numpy(y_train)
Xt_val, Xt_test = torch.from_numpy(X_val), torch.from_numpy(X_test)

### 11.1 scikit-learn

In [15]:
sk_model = LogisticRegression(C=C, max_iter=10_000, tol=1e-8)
sk_model.fit(X_train, y_train)
print("iterations:", sk_model.n_iter_[0])

iterations: 28


### 11.2 Manual PyTorch loop

Raw tensors with `requires_grad`, the loss written by hand, `backward()` for the
gradients, and a manual update `w <- w - lr * grad`. The loss uses the numerically
stable form `log(1 + e^z) - y*z` (= binary cross-entropy written in terms of the logit),
computed with `softplus`, so we never take `log(0)`.

In [16]:
w = torch.zeros(N_FEATURES, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
manual_history = []

for epoch in range(EPOCHS):
    z = Xt_train @ w + b
    bce = (torch.nn.functional.softplus(z) - yt_train * z).mean()
    loss = bce + 0.5 * LAMBDA * (w ** 2).sum()

    loss.backward()
    with torch.no_grad():
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad.zero_()
    b.grad.zero_()
    manual_history.append(loss.item())

print(f"final loss: {manual_history[-1]:.6f}")

final loss: 0.232348


### 11.3 Standard PyTorch workflow (`nn.Module` + `torch.optim`)

The same model as a `nn.Linear(n_features, 1)` layer, `BCEWithLogitsLoss`, and
`torch.optim.SGD` doing the update. The L2 penalty is the optimiser's
`weight_decay = λ`, applied to the weights only (a separate parameter group for the
bias, with no decay), so the objective matches the other two.

In [17]:
class LogisticRegressionNet(torch.nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)  # logits; the sigmoid lives in the loss


torch_model = LogisticRegressionNet(N_FEATURES)
torch.nn.init.zeros_(torch_model.linear.weight)  # same starting point as the manual loop
torch.nn.init.zeros_(torch_model.linear.bias)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([
    {"params": torch_model.linear.weight, "weight_decay": LAMBDA},
    {"params": torch_model.linear.bias, "weight_decay": 0.0},
], lr=LR)

standard_history = []
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    loss = criterion(torch_model(Xt_train), yt_train)
    loss.backward()
    optimizer.step()
    # log the same penalised objective as the manual loop, for a like-for-like curve
    with torch.no_grad():
        penalty = 0.5 * LAMBDA * (torch_model.linear.weight ** 2).sum().item()
    standard_history.append(loss.item() + penalty)

print(f"final loss: {standard_history[-1]:.6f}")

final loss: 0.232349


## 12. Do the three implementations agree?

In [18]:
weights = pd.DataFrame({
    "sklearn": np.r_[sk_model.intercept_, sk_model.coef_[0]],
    "manual_torch": np.r_[b.detach().numpy(), w.detach().numpy()],
    "standard_torch": np.r_[torch_model.linear.bias.detach().numpy(),
                            torch_model.linear.weight.detach().numpy()[0]],
}, index=["intercept"] + FEATURE_NAMES)


def predict_all(X, Xt):
    with torch.no_grad():
        return {
            "sklearn": sk_model.predict_proba(X)[:, 1],
            "manual_torch": torch.sigmoid(Xt @ w + b).numpy(),
            "standard_torch": torch.sigmoid(torch_model(Xt)).numpy(),
        }


proba = {"val": predict_all(X_val, Xt_val), "test": predict_all(X_test, Xt_test)}

for other in ["manual_torch", "standard_torch"]:
    dw = (weights[other] - weights["sklearn"]).abs().max()
    dp = np.abs(proba["test"][other] - proba["test"]["sklearn"]).max()
    same_calls = (select_top_per_period(proba["test"][other], test_df["period"])
                  == select_top_per_period(proba["test"]["sklearn"], test_df["period"])).mean()
    print(f"{other:15s} vs sklearn: max |weight diff| = {dw:.5f} | max |proba diff| (test) = {dp:.5f} "
          f"| same call decision for {same_calls:.2%} of test clients")

print()
print("Largest weights (standardised features):")
print(weights.reindex(weights["sklearn"].abs().sort_values(ascending=False).index).head(10).round(4))

manual_torch    vs sklearn: max |weight diff| = 0.00453 | max |proba diff| (test) = 0.00084 | same call decision for 100.00% of test clients
standard_torch  vs sklearn: max |weight diff| = 0.00453 | max |proba diff| (test) = 0.00084 | same call decision for 100.00% of test clients

Largest weights (standardised features):
                                  sklearn  manual_torch  standard_torch
intercept                         -2.7917       -2.7916         -2.7916
num__euribor3m                    -0.3707       -0.3707         -0.3707
cat__poutcome_nonexistent          0.2603        0.2601          0.2601
cat__contact_telephone            -0.1869       -0.1868         -0.1868
cat__poutcome_success              0.1622        0.1623          0.1623
cat__education_university.degree   0.1188        0.1181          0.1181
num__log_campaign                 -0.1035       -0.1034         -0.1034
cat__default_unknown              -0.1031       -0.1031         -0.1031
cat__age_band_41-50         

## 13. Results: val and test, against the baseline

The test set is used only here, once, after every decision was made on validation.

In [19]:
rows = []
for split_name, frame, y in [("val", val_df, y_val), ("test", test_df, y_test)]:
    rows.append({"split": split_name, "model": "baseline (random 25%)", **baseline[split_name]})
    for model_name, p in proba[split_name].items():
        rows.append({"split": split_name, "model": model_name, **evaluate(y, p, frame["period"])})

results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))

split                 model  precision@25%   lift  recall@25%  business_cost_eur    auc  log_loss
  val baseline (random 25%)         0.3691 0.9445      0.2361              88990 0.5000    1.0980
  val               sklearn         0.5317 1.3604      0.3401              76600 0.6034    0.7442
  val          manual_torch         0.5317 1.3604      0.3401              76600 0.6034    0.7441
  val        standard_torch         0.5317 1.3604      0.3401              76600 0.6034    0.7441
 test baseline (random 25%)         0.5087 0.9758      0.2442              82365 0.5000    1.4416
 test               sklearn         0.7592 1.4562      0.3644              68820 0.6981    0.7948
 test          manual_torch         0.7592 1.4562      0.3644              68820 0.6981    0.7947
 test        standard_torch         0.7592 1.4562      0.3644              68820 0.6981    0.7947


## 14. Save artifacts for the dashboard

The dashboard loads these files and never retrains. The default threshold for its
slider is the probability that selects 25% of the **validation** clients.

In [20]:
import json

joblib.dump(sk_model, ARTIFACTS / "sklearn_model.joblib")
torch.save({"w": w.detach(), "b": b.detach()}, ARTIFACTS / "manual_torch_model.pt")
torch.save(torch_model.state_dict(), ARTIFACTS / "standard_torch_model.pt")

default_threshold = float(np.quantile(proba["val"]["sklearn"], 1 - CAPACITY))
metadata = {
    "categorical": CATEGORICAL,
    "numeric": NUMERIC,
    "feature_names": FEATURE_NAMES,
    "n_features": N_FEATURES,
    "capacity": CAPACITY,
    "cost_fp": COST_FP,
    "cost_fn": COST_FN,
    "default_threshold": default_threshold,
    "hyperparameters": {"C": C, "lr": LR, "epochs": EPOCHS, "lambda": LAMBDA},
    "loss_history": {"manual_torch": manual_history, "standard_torch": standard_history},
}
(ARTIFACTS / "metadata.json").write_text(json.dumps(metadata, indent=2))
results.to_csv(ARTIFACTS / "results.csv", index=False)
weights.to_csv(ARTIFACTS / "weights.csv")
print(f"Default threshold (25% of validation called): {default_threshold:.3f}")
print("Saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

Default threshold (25% of validation called): 0.316
Saved: ['dataset_with_split.csv', 'manual_torch_model.pt', 'metadata.json', 'preprocessor.joblib', 'results.csv', 'sklearn_model.joblib', 'standard_torch_model.pt', 'weights.csv']
